# 1. Ingestão dos Dados - Camada Bronze

Nesta primeira etapa do pipeline, são utilizados os arquivos de Dotação e Execução Orçamentária da CAPES referentes ao período entre 2013 e 2020. Inicialmente, os oito arquivos em formato CSV são carregados e reunidos em um único DataFrame, permitindo concentrar os dados dos diferentes anos em uma mesma estrutura. Em seguida, os dados são armazenados na camada Bronze em formato Delta, buscando preservar as informações originais disponibilizadas pela CAPES. Além disso, são adicionadas informações referentes à origem dos arquivos e à data de ingestão, possibilitando maior rastreabilidade dos dados ao longo do pipeline.

##1.1 Orçamento Capes

In [0]:
%sql

USE CATALOG mvp;
USE SCHEMA bronze;

In [0]:
# Define o caminho onde estão armazenados os arquivos CSV da CAPES
caminho = "/Volumes/workspace/default/capes_dados/*.csv"

In [0]:
# Lendo os dados do arquivo
df_bronze = (
    spark.read
    .option("header", "true") #Lê a primeira linha como cabeçalho
    .option("sep", ";") # Separação por ;
    .option("inferSchema", "false") # Mantém os campos como string na camada bronze; as conversões serão realizadas na Silver
    .csv(caminho) # lê os arquivos csv
)

In [0]:
# Adiciona metadados técnicos de rastreabilidade à camada Bronze.
from pyspark.sql.functions import current_timestamp, col

df_bronze = (
    df_bronze
    .withColumn("ARQUIVO_ORIGEM", col("_metadata.file_path"))
    .withColumn("DT_INGESTAO", current_timestamp())
)

In [0]:
# Disponibiliza o DataFrame para consultas SQL
df_bronze.createOrReplaceTempView("vw_bronze_capes")

In [0]:
%sql

SELECT
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_capes;

QTD_REGISTROS
15739


In [0]:
%sql

SELECT
    AN_ORCAMENTO,
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_capes
GROUP BY AN_ORCAMENTO
ORDER BY AN_ORCAMENTO;

AN_ORCAMENTO,QTD_REGISTROS
2013,2422
2014,2211
2015,1774
2016,1952
2017,1956
2018,2032
2019,1888
2020,1504


In [0]:
%sql

CREATE OR REPLACE TABLE mvp.bronze.orcamento_capes
USING DELTA
AS

SELECT *
FROM vw_bronze_capes;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Valida a presença dos campos técnicos de rastreabilidade.

SELECT
    AN_ORCAMENTO,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM mvp.bronze.orcamento_capes
LIMIT 10;

AN_ORCAMENTO,ARQUIVO_ORIGEM,DT_INGESTAO
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z


In [0]:
%sql

SELECT *
FROM mvp.bronze.orcamento_capes
LIMIT 10;

AN_ORCAMENTO,CD_UNIDADE_GESTORA,DS_UNIDADE_GESTORA,CD_FUNCAO_ORCAMENTARIA,DS_FUNCAO_ORCAMENTARIA,CD_SUBFUNCAO_ORCAMENTARIA,DS_SUBFUNCAO_ORCAMENTARIA,CD_PROGRAMA_GOVERNO,DS_PROGRAMA_GOVERNO,CD_ACAO,DS_ACAO,CD_PLANO_ORCAMENTARIO,DS_PLANO_ORCAMENTARIO,DS_PROGRAMA_TRABALHO,CD_PTRES,CD_PLANO_INTERNO,DS_PLANO_INTERNO,CD_MODALIDADE_APLICACAO,DS_MODALIDADE_APLICACAO,CD_CATEGORIA_ECONOMICA,DS_CATEGORIA_ECONOMICA,CD_GRUPO_DESPESA,DS_GRUPO_DESPESA,CD_ELEMENTO_DESPESA,DS_ELEMENTO_DESPESA,CD_NATUREZA_DESPESA,DS_NATUREZA_DESPESA,CD_FONTE_RECURSO,DS_FONTE_RECURSO,VL_DOTACAO_INICIAL,VL_DOTACAO_SUPLEMENTAR,VL_DOTACAO_DISPONIVEL,VL_CREDITO_BLOQUEADO,VL_DESPESAS_EMPENHADAS,VL_EM_A_LIQUIDAR,VL_EM_LIQUIDADO,VL_DESPESAS_PAGAS,ARQUIVO_ORIGEM,DT_INGESTAO
2013,113201,COMISSAO NACIONAL DE ENERGIA NUCLEAR,12,EDUCACAO,364,ENSINO SUPERIOR,2032,"EDUCACAO SUPERIOR - GRADUACAO, POS-GRADUACAO, ENSINO, PESQUI",0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364203204870001,061720,PCC11O94PPN,PROAP - INSTITUCIONAL,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,33,PASSAGENS E DESPESAS COM LOCOMOCAO,339033,PASSAGENS E DESPESAS COM LOCOMOCAO,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,null,null,null,null,"38,131.95","24,578.70","13,553.25","13,553.25",dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,113203,COMISSAO NACIONAL DE ENERGIA NUCLEAR-IEN,12,EDUCACAO,364,ENSINO SUPERIOR,2032,"EDUCACAO SUPERIOR - GRADUACAO, POS-GRADUACAO, ENSINO, PESQUI",0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364203204870001,061720,PCC11O94PPN,PROAP - INSTITUCIONAL,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,14,DIARIAS - PESSOAL CIVIL,339014,DIARIAS - PESSOAL CIVIL,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,null,null,null,null,"5,676.24","1,419.06","4,257.18","4,257.18",dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,113203,COMISSAO NACIONAL DE ENERGIA NUCLEAR-IEN,12,EDUCACAO,364,ENSINO SUPERIOR,2032,"EDUCACAO SUPERIOR - GRADUACAO, POS-GRADUACAO, ENSINO, PESQUI",0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364203204870001,061720,PCC11O94PPN,PROAP - INSTITUCIONAL,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,39,OUTROS SERVICOS DE TERCEIROS PJ - OP.INT.ORC.,339039,OUTROS SERVICOS DE TERCEIROS - PESSOA JURIDICA,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,null,null,null,null,"33,572.28","6,000.00","27,572.28","27,572.28",dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,113204,COMISSAO NACIONAL DE ENERGIA NUCLEAR-IRD,12,EDUCACAO,364,ENSINO SUPERIOR,2032,"EDUCACAO SUPERIOR - GRADUACAO, POS-GRADUACAO, ENSINO, PESQUI",0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364203204870001,061720,PCC11O94PPN,PROAP - INSTITUCIONAL,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,36,OUTROS SERVICOS DE TERCEIROS - P.FISICA,339036,OUTROS SERVICOS DE TERCEIROS - PESSOA FISICA,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,null,null,null,null,"34,600.00","4,200.00","15,400.00","15,400.00",dbfs:/Volumes/workspace/default/capes_dados/capes_2013.csv,2026-09-18T19:59:11.203Z
2013,113204,COMISSAO NACIONAL DE ENERGIA NUCLEAR-IRD,12,EDUCACAO,364,ENSINO SUPERIOR,2032,"EDUCACAO SUPERIOR - GRADUACAO, POS-GRADUACAO, ENSINO, PESQUI",0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364203204870001,061720,PCC11O94PPN,PROAP - INSTITUCIONAL,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,39,OUTROS SERVICOS DE TERCEIROS PJ - OP.INT.ORC.,339039,OUTROS SERVICOS DE TERCEIROS - PESSOA JURIDICA,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,null,null,null,null,"150,795.00","29,240.00","116,080.00","113,980.00",dbfs:/Volumes/workspace/default/capes_dad

In [0]:
%sql

SELECT
    AN_ORCAMENTO,
    COUNT(*) AS QTD_REGISTROS
FROM mvp.bronze.orcamento_capes
GROUP BY AN_ORCAMENTO
ORDER BY AN_ORCAMENTO;

AN_ORCAMENTO,QTD_REGISTROS
2013,2422
2014,2211
2015,1774
2016,1952
2017,1956
2018,2032
2019,1888
2020,1504


## 1.2 Orçamento do MEC por Órgão Orçamentário

In [0]:
# Define o caminho do arquivo de orçamento do MEC
caminho_mec = "/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv"

In [0]:
# Realiza a leitura do arquivo de orçamento do MEC
df_mec_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ";")            # O arquivo utiliza ; como separador
    .option("inferSchema", "false") # Mantém os campos como string na Bronze
    .csv(caminho_mec)
)

display(df_mec_bronze)

Ano,Órgão Orçamentário,Projeto de Lei,Dotação Inicial,Dotação Atual,Empenhado,Liquidado,Pago
Total,null,2.428.601.478.704,2.216.507.005.011,2.399.310.360.960,2.270.713.386.859,2.000.178.707.035,1.921.119.548.959
2000,26000 - Ministério da Educação,12.389.063.938,12.638.403.595,14.359.294.228,13.385.729.242,13.385.729.242,11.541.552.456
2001,26000 - Ministério da Educação,15.423.795.227,16.163.900.856,16.760.032.191,14.485.011.306,14.485.011.306,13.234.679.546
2002,26000 - Ministério da Educação,16.727.181.308,17.421.387.553,18.261.210.134,16.659.309.712,16.659.309.712,15.760.729.579
2003,26000 - Ministério da Educação,17.697.458.106,18.037.343.186,18.902.857.426,18.118.944.472,18.117.050.992,16.570.771.043
2004,26000 - Ministério da Educação,18.817.972.991,17.303.144.820,19.521.090.512,18.388.018.034,18.388.018.034,17.157.878.368
2005,26000 - Ministério da Educação,20.701.591.606,21.022.574.093,22.329.424.928,20.028.496.794,20.028.496.794,18.082.501.460
2006,26000 - Ministério da Educação,21.295.491.099,21.671.079.236,24.904.363.112,23.925.513.623,23.925.513.474,21.673.566.266
2007,26000 - Ministério da Educação,26.401.103.572,27.579.847.716,29.474.812.754,28.713.160.322,26.020.635.780,24.625.885.394
2008,26000 - Ministério da Educação,31.246.828.910,31.714.041.624,34.820.102.236,33.014.256.077,29.854.938.336,28.984.386.427


In [0]:
# Adiciona metadados técnicos de rastreabilidade à camada Bronze
from pyspark.sql.functions import current_timestamp, lit

df_mec_bronze = (
    df_mec_bronze
    .withColumn("ARQUIVO_ORIGEM", lit(caminho_mec))
    .withColumn("DT_INGESTAO", current_timestamp())
)

In [0]:
# Disponibiliza o DataFrame para consultas em SQL
df_mec_bronze.createOrReplaceTempView("vw_bronze_mec")

In [0]:
%sql

SELECT
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_mec;

QTD_REGISTROS
29


In [0]:
# Exibe a estrutura dos dados
df_mec_bronze.printSchema()

root
 |-- Ano: string (nullable = true)
 |-- Órgão Orçamentário: string (nullable = true)
 |-- Projeto de Lei: string (nullable = true)
 |-- Dotação Inicial: string (nullable = true)
 |-- Dotação Atual: string (nullable = true)
 |-- Empenhado: string (nullable = true)
 |-- Liquidado: string (nullable = true)
 |-- Pago: string (nullable = true)
 |-- ARQUIVO_ORIGEM: string (nullable = false)
 |-- DT_INGESTAO: timestamp (nullable = false)



In [0]:
%sql
-- Exibe os campos de rastreabilidade adicionados

SELECT
    `Ano`,
    `Órgão Orçamentário`,
    ARQUIVO_ORIGEM,
    DT_INGESTAO

FROM vw_bronze_mec

LIMIT 20;

Ano,Órgão Orçamentário,ARQUIVO_ORIGEM,DT_INGESTAO
Total,null,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2000,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2001,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2002,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2003,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2004,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2005,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2006,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2007,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z
2008,26000 - Ministério da Educação,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:17.135Z


In [0]:
%sql

-- Padroniza os nomes das colunas para permitir a persistência em Delta.

CREATE OR REPLACE TEMP VIEW vw_bronze_mec_padronizado AS

SELECT
    `Ano` AS ANO,
    `Órgão Orçamentário` AS ORGAO_ORCAMENTARIO,
    `Projeto de Lei` AS PROJETO_LEI,
    `Dotação Inicial` AS DOTACAO_INICIAL,
    `Dotação Atual` AS DOTACAO_ATUAL,
    `Empenhado` AS EMPENHADO,
    `Liquidado` AS LIQUIDADO,
    `Pago` AS PAGO,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM vw_bronze_mec;

In [0]:
%sql

-- Persiste os dados padronizados do MEC como tabela Delta na camada Bronze.

CREATE OR REPLACE TABLE mvp.bronze.orcamento_mec
USING DELTA
AS

SELECT *
FROM vw_bronze_mec_padronizado;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Valida a tabela persistida na camada Bronze.

SELECT *
FROM mvp.bronze.orcamento_mec
ORDER BY ANO
LIMIT 10;

ANO,ORGAO_ORCAMENTARIO,PROJETO_LEI,DOTACAO_INICIAL,DOTACAO_ATUAL,EMPENHADO,LIQUIDADO,PAGO,ARQUIVO_ORIGEM,DT_INGESTAO
2000,26000 - Ministério da Educação,12.389.063.938,12.638.403.595,14.359.294.228,13.385.729.242,13.385.729.242,11.541.552.456,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2001,26000 - Ministério da Educação,15.423.795.227,16.163.900.856,16.760.032.191,14.485.011.306,14.485.011.306,13.234.679.546,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2002,26000 - Ministério da Educação,16.727.181.308,17.421.387.553,18.261.210.134,16.659.309.712,16.659.309.712,15.760.729.579,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2003,26000 - Ministério da Educação,17.697.458.106,18.037.343.186,18.902.857.426,18.118.944.472,18.117.050.992,16.570.771.043,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2004,26000 - Ministério da Educação,18.817.972.991,17.303.144.820,19.521.090.512,18.388.018.034,18.388.018.034,17.157.878.368,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2005,26000 - Ministério da Educação,20.701.591.606,21.022.574.093,22.329.424.928,20.028.496.794,20.028.496.794,18.082.501.460,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2006,26000 - Ministério da Educação,21.295.491.099,21.671.079.236,24.904.363.112,23.925.513.623,23.925.513.474,21.673.566.266,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2007,26000 - Ministério da Educação,26.401.103.572,27.579.847.716,29.474.812.754,28.713.160.322,26.020.635.780,24.625.885.394,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2008,26000 - Ministério da Educação,31.246.828.910,31.714.041.624,34.820.102.236,33.014.256.077,29.854.938.336,28.984.386.427,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z
2009,26000 - Ministério da Educação,41.560.934.226,40.524.634.534,45.283.180.939,41.798.912.405,37.380.465.734,36.388.742.029,/Volumes/workspace/default/mec_orcamento/MEC_Orcamento.csv,2026-09-18T20:02:21.376Z



# 1.3 Orçamento do MEC por Unidade Orçamentária

In [0]:
# Define o caminho do arquivo de orçamento do MEC por Unidade Orçamentária
caminho_mec_unidades = "/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv"

In [0]:
# Realiza a leitura do arquivo de orçamento do MEC por Unidade Orçamentária
df_mec_unidades_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ";")             # O arquivo utiliza ; como separador
    .option("inferSchema", "false") # Mantém os campos como string na Bronze
    .csv(caminho_mec_unidades)
)

display(df_mec_unidades_bronze)

Ano,Unidade Orçamentária,Projeto de Lei,Dotação Inicial,Dotação Atual,Empenhado,Liquidado,Pago
Total,null,2.428.601.478.704,2.216.507.005.011,2.399.310.360.960,2.270.713.386.859,2.000.178.707.035,1.921.119.548.959
2000,26101 - Ministério da Educação - Administração Direta,934.768.645,1.011.675.165,1.517.602.292,1.095.324.226,1.095.324.226,768.876.660
2000,26104 - Instituto Nacional de Educação de Surdos,18.313.544,18.363.544,19.288.725,19.149.370,19.149.370,18.573.767
2000,26105 - Instituto Benjamin Constant,13.198.636,13.198.636,14.209.434,14.062.175,14.062.175,13.817.734
2000,26201 - Colégio Pedro II,96.107.370,96.507.370,100.894.215,100.565.307,100.565.307,95.958.608
2000,26202 - Centro Federal de Educação Tecnológica de Alagoas,22.909.795,22.909.795,25.866.340,25.607.522,25.607.522,24.962.915
2000,26203 - Centro Federal de Educação Tecnológica do Amazonas,16.423.073,16.573.073,18.708.071,18.149.009,18.149.009,16.694.736
2000,26205 - Centro Federal de Educação Tecnológica de Campos,25.012.855,25.087.855,26.791.709,26.599.054,26.599.054,26.440.135
2000,26206 - Centro Federal de Educação Tecnológica do Ceará,19.978.656,20.058.656,22.238.271,22.162.137,22.162.137,21.616.907
2000,26207 - Centro Federal de Educação Tecnológica do Espírito Santo,19.430.094,19.430.094,23.030.362,21.633.121,21.633.121,21.467.282


In [0]:
# Adiciona metadados técnicos de rastreabilidade à camada Bronze
from pyspark.sql.functions import current_timestamp, lit

df_mec_unidades_bronze = (
    df_mec_unidades_bronze
    .withColumn("ARQUIVO_ORIGEM", lit(caminho_mec_unidades))
    .withColumn("DT_INGESTAO", current_timestamp())
)

In [0]:
# Disponibiliza o DataFrame para consultas em SQL
df_mec_unidades_bronze.createOrReplaceTempView(
    "vw_bronze_mec_unidades"
)

In [0]:
%sql

-- Valida a quantidade de registros carregados.

SELECT
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_mec_unidades;

QTD_REGISTROS
4060


In [0]:
%sql

-- Exibe os campos técnicos adicionados durante a ingestão.

SELECT
    `Ano`,
    `Unidade Orçamentária`,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM vw_bronze_mec_unidades
LIMIT 20;

Ano,Unidade Orçamentária,ARQUIVO_ORIGEM,DT_INGESTAO
Total,null,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26101 - Ministério da Educação - Administração Direta,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26104 - Instituto Nacional de Educação de Surdos,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26105 - Instituto Benjamin Constant,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26201 - Colégio Pedro II,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26202 - Centro Federal de Educação Tecnológica de Alagoas,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26203 - Centro Federal de Educação Tecnológica do Amazonas,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26205 - Centro Federal de Educação Tecnológica de Campos,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26206 - Centro Federal de Educação Tecnológica do Ceará,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z
2000,26207 - Centro Federal de Educação Tecnológica do Espírito Santo,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:29.569Z


In [0]:
%sql

-- Padroniza os nomes das colunas para permitir a persistência em Delta.

CREATE OR REPLACE TEMP VIEW vw_bronze_mec_unidades_padronizado AS

SELECT
    `Ano` AS ANO,
    `Unidade Orçamentária` AS UNIDADE_ORCAMENTARIA,
    `Projeto de Lei` AS PROJETO_LEI,
    `Dotação Inicial` AS DOTACAO_INICIAL,
    `Dotação Atual` AS DOTACAO_ATUAL,
    `Empenhado` AS EMPENHADO,
    `Liquidado` AS LIQUIDADO,
    `Pago` AS PAGO,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM vw_bronze_mec_unidades;

In [0]:
%sql

-- Persiste os dados do MEC por Unidade Orçamentária na camada Bronze.

CREATE OR REPLACE TABLE mvp.bronze.orcamento_mec_unidades
USING DELTA
AS

SELECT *
FROM vw_bronze_mec_unidades_padronizado;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Valida a tabela persistida na camada Bronze.

SELECT *
FROM mvp.bronze.orcamento_mec_unidades
LIMIT 10;

ANO,UNIDADE_ORCAMENTARIA,PROJETO_LEI,DOTACAO_INICIAL,DOTACAO_ATUAL,EMPENHADO,LIQUIDADO,PAGO,ARQUIVO_ORIGEM,DT_INGESTAO
Total,null,2.428.601.478.704,2.216.507.005.011,2.399.310.360.960,2.270.713.386.859,2.000.178.707.035,1.921.119.548.959,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26101 - Ministério da Educação - Administração Direta,934.768.645,1.011.675.165,1.517.602.292,1.095.324.226,1.095.324.226,768.876.660,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26104 - Instituto Nacional de Educação de Surdos,18.313.544,18.363.544,19.288.725,19.149.370,19.149.370,18.573.767,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26105 - Instituto Benjamin Constant,13.198.636,13.198.636,14.209.434,14.062.175,14.062.175,13.817.734,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26201 - Colégio Pedro II,96.107.370,96.507.370,100.894.215,100.565.307,100.565.307,95.958.608,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26202 - Centro Federal de Educação Tecnológica de Alagoas,22.909.795,22.909.795,25.866.340,25.607.522,25.607.522,24.962.915,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26203 - Centro Federal de Educação Tecnológica do Amazonas,16.423.073,16.573.073,18.708.071,18.149.009,18.149.009,16.694.736,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26205 - Centro Federal de Educação Tecnológica de Campos,25.012.855,25.087.855,26.791.709,26.599.054,26.599.054,26.440.135,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26206 - Centro Federal de Educação Tecnológica do Ceará,19.978.656,20.058.656,22.238.271,22.162.137,22.162.137,21.616.907,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z
2000,26207 - Centro Federal de Educação Tecnológica do Espírito Santo,19.430.094,19.430.094,23.030.362,21.633.121,21.633.121,21.467.282,/Volumes/workspace/default/mec_unidades/MEC_UnidadeOrcamentaria.csv,2026-09-18T20:02:33.180Z


In [0]:
%sql

-- Verifica a presença da CAPES entre as unidades orçamentárias do MEC.

SELECT
    ANO,
    UNIDADE_ORCAMENTARIA,
    PROJETO_LEI,
    DOTACAO_INICIAL,
    DOTACAO_ATUAL,
    EMPENHADO,
    LIQUIDADO,
    PAGO
FROM mvp.bronze.orcamento_mec_unidades
WHERE UNIDADE_ORCAMENTARIA LIKE '26291%'
ORDER BY ANO;

ANO,UNIDADE_ORCAMENTARIA,PROJETO_LEI,DOTACAO_INICIAL,DOTACAO_ATUAL,EMPENHADO,LIQUIDADO,PAGO
2000,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,450.365.404,450.365.404,458.240.295,453.873.970,453.873.970,367.825.677
2001,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,456.037.138,467.127.476,507.869.850,500.082.874,500.082.874,403.937.569
2002,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,475.325.696,475.062.463,515.513.311,520.424.121,520.424.121,399.354.064
2003,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,519.532.210,520.002.210,545.438.860,535.804.331,535.708.231,414.533.843
2004,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,520.274.646,520.274.646,579.107.143,566.227.674,566.227.674,493.995.778
2005,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,683.190.604,676.773.137,670.240.058,657.566.382,657.566.382,613.261.650
2006,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,727.182.166,725.630.587,745.945.328,734.738.981,734.738.981,656.921.420
2007,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,789.936.800,789.936.800,816.564.660,781.238.545,695.738.108,694.379.872
2008,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,1.276.199.972,1.254.699.972,1.275.548.817,1.260.604.503,1.017.512.021,958.032.180
2009,26291 - Fundação Coordenação de Aperfeiçoamento de Pessoal de Nível Superior,2.055.361.380,1.432.246.991,1.927.926.074,1.622.761.267,1.297.389.304,1.203.936.715


# 1.4 Número de Bolsas distribuídas pelo CAPES

In [0]:
# Define o caminho dos arquivos de concessão de bolsas da CAPES
caminho_bolsas = "/Volumes/workspace/default/capes_bolsas/*.csv"

In [0]:
# Realiza a leitura dos arquivos de concessão de bolsas da CAPES
df_bolsas_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("inferSchema", "false")  # Mantém os campos como string na Bronze
    .csv(caminho_bolsas)
)

display(df_bolsas_bronze)

AN_REFERENCIA,NM_MUNICIPIO_IES_CAPES,SG_UF_IES_CAPES,NM_REGIAO_IES_CAPES,CD_IES_CAPES,NM_IES_CAPES,SG_IES_CAPES,NM_NATUREZA_JURIDICA,SG_PROGRAMA_FOMENTO,CD_PROGRAMA,NM_PROGRAMA,NM_GRANDE_AREA,NM_AREA_AVALIACAO,NM_AREA_CONHECIMENTO,NM_NIVEL_PADRAO,QT_BOLSAS_CONCEDIDAS
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016003P1,CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,FARM�CIA,DOUTORADO,8
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016003P1,CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,FARM�CIA,MESTRADO,13
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016004P8,QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,QU�MICA,DOUTORADO,9
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016004P8,QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,QU�MICA,MESTRADO,17
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016005P4,CI�NCIAS AMBIENTAIS,MULTIDISCIPLINAR,CI�NCIAS AMBIENTAIS,CI�NCIAS AMBIENTAIS,MESTRADO,11
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016006P0,ENFERMAGEM,CI�NCIAS DA SA�DE,ENFERMAGEM,ENFERMAGEM,MESTRADO,8
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016007P7,Bioci�ncias Aplicadas � Sa�de,CI�NCIAS DA SA�DE,MEDICINA II,ANATOMIA PATOL�GICA E PATOLOGIA CL�NICA,DOUTORADO,6
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016007P7,Bioci�ncias Aplicadas � Sa�de,CI�NCIAS DA SA�DE,MEDICINA II,ANATOMIA PATOL�GICA E PATOLOGIA CL�NICA,MESTRADO,6
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016008P3,CI�NCIA E ENGENHARIA DE MATERIAIS,ENGENHARIAS,ENGENHARIAS II,ENGENHARIA DE MATERIAIS E METAL�RGICA,MESTRADO,6
2018,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016009P0,CI�NCIAS ODONTOL�GICAS,CI�NCIAS DA SA�DE,ODONTOLOGIA,ODONTOLOGIA,MESTRADO,5


In [0]:
# Valida a quantidade de registros e atributos carregados
print("Quantidade de registros:", df_bolsas_bronze.count())
print("Quantidade de colunas:", len(df_bolsas_bronze.columns))

Quantidade de registros: 103360
Quantidade de colunas: 16


In [0]:
# Adiciona metadados técnicos de rastreabilidade à camada Bronze
from pyspark.sql.functions import current_timestamp, col

df_bolsas_bronze = (
    df_bolsas_bronze
    .withColumn("ARQUIVO_ORIGEM", col("_metadata.file_path"))
    .withColumn("DT_INGESTAO", current_timestamp())
)

In [0]:
# Disponibiliza o DataFrame para consultas em SQL
df_bolsas_bronze.createOrReplaceTempView(
    "vw_bronze_bolsas"
)

In [0]:
%sql

-- Valida a quantidade de registros carregados.

SELECT
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_bolsas;

QTD_REGISTROS
103360


In [0]:
print("Quantidade de colunas:", len(df_bolsas_bronze.columns))

Quantidade de colunas: 18


In [0]:
%sql

-- Verifica a quantidade de registros provenientes de cada arquivo.

SELECT
    ARQUIVO_ORIGEM,
    COUNT(*) AS QTD_REGISTROS
FROM vw_bronze_bolsas
GROUP BY ARQUIVO_ORIGEM
ORDER BY ARQUIVO_ORIGEM;

ARQUIVO_ORIGEM,QTD_REGISTROS
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,12168
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2014.csv,12368
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2015.csv,13117
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2016.csv,12742
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2017.csv,13660
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2018.csv,14092
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2019.csv,13206
dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2020.csv,12007


In [0]:
%sql

-- Persiste os dados de bolsas da CAPES como uma tabela Delta na camada Bronze.

CREATE OR REPLACE TABLE mvp.bronze.bolsas_capes
USING DELTA
AS
SELECT *
FROM vw_bronze_bolsas;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Valida os primeiros registros da tabela persistida na camada Bronze.

SELECT *
FROM mvp.bronze.bolsas_capes
LIMIT 10;

AN_REFERENCIA,NM_MUNICIPIO_IES_CAPES,SG_UF_IES_CAPES,NM_REGIAO_IES_CAPES,CD_IES_CAPES,NM_IES_CAPES,SG_IES_CAPES,NM_NATUREZA_JURIDICA,SG_PROGRAMA_FOMENTO,CD_PROGRAMA,NM_PROGRAMA,NM_GRANDE_AREA,NM_AREA_AVALIACAO,NM_AREA_CONHECIMENTO,NM_NIVEL_PADRAO,QT_BOLSAS_CONCEDIDAS,ARQUIVO_ORIGEM,DT_INGESTAO
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016003P1,CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,FARM�CIA,DOUTORADO,4,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016003P1,CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,FARM�CIA,MESTRADO,13,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016004P8,QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,QU�MICA,DOUTORADO,4,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016004P8,QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,QU�MICA,MESTRADO,16,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016005P4,CI�NCIAS AMBIENTAIS,MULTIDISCIPLINAR,CI�NCIAS AMBIENTAIS,CI�NCIAS AMBIENTAIS,MESTRADO,11,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016006P0,ENFERMAGEM,CI�NCIAS DA SA�DE,ENFERMAGEM,ENFERMAGEM,MESTRADO,6,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016007P7,Bioci�ncias Aplicadas � Sa�de,CI�NCIAS DA SA�DE,MEDICINA II,ANATOMIA PATOL�GICA E PATOLOGIA CL�NICA,MESTRADO,7,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016008P3,CI�NCIA E ENGENHARIA DE MATERIAIS,ENGENHARIAS,ENGENHARIAS II,ENGENHARIA DE MATERIAIS E METAL�RGICA,MESTRADO,4,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016009P0,CI�NCIAS ODONTOL�GICAS,CI�NCIAS DA SA�DE,ODONTOLOGIA,ODONTOLOGIA,MESTRADO,3,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z
2013,ALFENAS,MG,SUDESTE,32011016,UNIVERSIDADE FEDERAL DE ALFENAS,UNIFAL-MG,FEDERAL,DS,32011016010P8,F�SICA - UFLA - UFSJ - UNIFAL,CI�NCIAS EXATAS E DA TERRA,ASTRONOMIA / F�SICA,F�SICA,MESTRADO,3,dbfs:/Volumes/workspace/default/capes_bolsas/bolsas_2013.csv,2026-09-18T20:02:48.820Z


In [0]:
%sql

-- Verifica o formato original dos valores monetários antes do tratamento da Silver.

SELECT
    AN_ORCAMENTO,
    CD_ACAO,
    CD_PLANO_INTERNO,
    DS_PLANO_INTERNO,
    VL_DOTACAO_INICIAL,
    VL_DOTACAO_DISPONIVEL,
    VL_DESPESAS_EMPENHADAS

FROM mvp.bronze.orcamento_capes

WHERE AN_ORCAMENTO = '2013'
  AND CD_UNIDADE_GESTORA = '154003'
  AND CD_ACAO = '0487'
  AND CD_PLANO_INTERNO = 'PCC08O06PBY';

AN_ORCAMENTO,CD_ACAO,CD_PLANO_INTERNO,DS_PLANO_INTERNO,VL_DOTACAO_INICIAL,VL_DOTACAO_DISPONIVEL,VL_DESPESAS_EMPENHADAS
2013,0487,PCC08O06PBY,PROEX - MESTRADO,161334000.00,161334000.00,161334000.00


In [0]:
%sql
SELECT
    NM_PROGRAMA,
    NM_GRANDE_AREA,
    NM_AREA_AVALIACAO,
    NM_NIVEL_PADRAO
FROM mvp.bronze.bolsas_capes
WHERE NM_PROGRAMA LIKE '%�%'
   OR NM_GRANDE_AREA LIKE '%�%'
   OR NM_AREA_AVALIACAO LIKE '%�%'
   OR NM_NIVEL_PADRAO LIKE '%�%'
LIMIT 20;

NM_PROGRAMA,NM_GRANDE_AREA,NM_AREA_AVALIACAO,NM_NIVEL_PADRAO
CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,DOUTORADO
CI�NCIAS FARMAC�UTICAS,CI�NCIAS DA SA�DE,FARM�CIA,MESTRADO
QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,DOUTORADO
QU�MICA,CI�NCIAS EXATAS E DA TERRA,QU�MICA,MESTRADO
CI�NCIAS AMBIENTAIS,MULTIDISCIPLINAR,CI�NCIAS AMBIENTAIS,MESTRADO
ENFERMAGEM,CI�NCIAS DA SA�DE,ENFERMAGEM,MESTRADO
Bioci�ncias Aplicadas � Sa�de,CI�NCIAS DA SA�DE,MEDICINA II,MESTRADO
CI�NCIA E ENGENHARIA DE MATERIAIS,ENGENHARIAS,ENGENHARIAS II,MESTRADO
CI�NCIAS ODONTOL�GICAS,CI�NCIAS DA SA�DE,ODONTOLOGIA,MESTRADO
F�SICA - UFLA - UFSJ - UNIFAL,CI�NCIAS EXATAS E DA TERRA,ASTRONOMIA / F�SICA,MESTRADO
